# 🏗️ Notebook 1: Stock Exchange — Requirements & Architecture

## 🛠️ Setup

```bash
cd 06-system-designs/stock-exchange
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## What we're designing

A minimal stock exchange. Traders submit buy/sell orders; the exchange matches them according to **price-time priority** and publishes trades + market data.

Core of the system: the **matching engine** — a single-threaded loop that owns the order book.

## Requirements

### Functional
- Submit limit order (price + qty).
- Cancel/modify order.
- Match buys and sells when prices cross.
- Publish trades + top-of-book ticks.

### Non-functional
- **Determinism** — same input → same output.
- **Low latency** — sub-ms matching.
- **Durability** — every order persisted before ack.

## Back-of-envelope

- 10k symbols × 100 orders/sec avg = 1M orders/s peak.
- Market data fan-out: 1 trade → 100k subscribers.
- Latency budget: 100µs inside matcher, ~1ms end-to-end.

## High-level architecture

```
  [Trader]
     │ FIX/WebSocket
     ▼
  ┌────────────┐
  │  Gateway   │ auth, rate limit
  └─────┬──────┘
        │ sequenced
        ▼
  ┌────────────┐     ┌──────────────┐
  │  Sequencer │────►│ Journal (WAL)│
  └─────┬──────┘     └──────────────┘
        ▼
  ┌────────────┐     ┌──────────────┐
  │ Matching   │────►│  Trade feed  │──► subscribers
  │  Engine    │     └──────────────┘
  └────────────┘
```

- **One matching engine per symbol** — single-threaded for ordering.
- **WAL before match** so we can replay after crash.

## Why these choices?

- Each service in the diagram owns one responsibility — easier to scale and reason about.
- Stateless services scale horizontally; stateful stores are chosen per access pattern.
- The next two notebooks zoom into the **data model + APIs** and one **deep-dive algorithm**.